In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
import os

# Định nghĩa đường dẫn lưu trữ trên Drive
drive_path = '/content/drive/MyDrive/ModelBiLSTM/'
# Tạo thư mục nếu chưa có
os.makedirs(drive_path, exist_ok=True)
print(f"Thư mục lưu trữ: {drive_path}")

Thư mục lưu trữ: /content/drive/MyDrive/ModelBiLSTM/


In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [16]:
X_train = pd.read_csv('X_train.csv')
Y_train = pd.read_csv('Y_train.csv')
X_test = pd.read_csv('X_test.csv')
X_val = pd.read_csv('X_val.csv')
Y_val = pd.read_csv('Y_val.csv')

In [17]:
print("X_train shape:", X_train.shape)
print("Y_train shape:", Y_train.shape)
print("X_val shape:", X_val.shape)
print("Y_val shape:", Y_val.shape)
print("X_test shape:", X_test.shape)

X_train shape: (51000, 38)
Y_train shape: (51000, 7)
X_val shape: (7200, 38)
Y_val shape: (7200, 7)
X_test shape: (38000, 38)


In [18]:
# Gộp dữ liệu để train (file train + file val)
X_full_train = pd.concat([X_train])
Y_full_train = pd.concat([Y_train])

# X_full_train.shape, Y_full_train.shape, X_full_train.columns, Y_full_train.columns

In [19]:
X_full_train.shape, Y_full_train.shape

((51000, 38), (51000, 7))

In [20]:


# Dòng 1: Lấy 2 cột từ dữ liệu
a1 = Y_full_train['attr_1'].values  # [1,1,2,3,3,3]
a4 = Y_full_train['attr_4'].values  # [5,5,6,7,7,8]

# Dòng 2: Tìm các cặp duy nhất
valid_pairs = set(zip(a1, a4))  # {(1,5), (2,6), (3,7), (3,8)}

# Dòng 3: Chuyển về 0-based cho loss
valid_combinations = [[x-1, y-1] for x, y in valid_pairs]

In [21]:
# Y_train  = Y_train.drop(columns=["id"])
# Y_val = Y_val.drop(columns=["id"])

# Loại bổ cột ID của file Y_full_train
Y_full_train = Y_full_train.drop(columns=["id"])

In [22]:
Y_full_train.shape, Y_full_train.columns

((51000, 6),
 Index(['attr_1', 'attr_2', 'attr_3', 'attr_4', 'attr_5', 'attr_6'], dtype='object'))

In [23]:
# "Đoạn này chuyển nhãn từ 1,2,3... về 0,1,2... để CrossEntropyLoss hiểu được ạ. Đồng thời lưu lại encoder để sau này decode kết quả dự đoán về đúng giá trị gốc."
# Do các attr của mk nó liên tục nên đưa về này đc, cái CrossEntropyLoss của mk thì nó ko hiểu đc nếu dữ liệu chạy từ 1
from sklearn.preprocessing import LabelEncoder

encoders = {}

Y_train_encoded = Y_full_train.copy()

for col in Y_full_train.columns:
    le = LabelEncoder()
    Y_train_encoded[col] = le.fit_transform(Y_full_train[col])
    encoders[col] = le

In [24]:
Y_train_encoded.head()

,attr_1,attr_2,attr_3,attr_4,attr_5,attr_6
0,2,7,51,2,7,10
1,10,0,5,10,0,44
2,2,30,23,4,18,68
3,10,0,87,11,0,97
4,2,20,57,11,14,93


In [25]:
# Lây ra cột id của X_test và X_val để sau khi dự đoán xong thực hiện ghép để xuất file
X_test_id = X_test["id"]

#### TIỀN XỬ LÝ DỮ LIỆU

Đoạn tiền xử lý dữ liệu này thì em đang làm như trên docs vs hôm anh Tuấn Anh có bảo

In [26]:
# Tiền xử lý dữ liệu
import numpy as np
import pandas as pd

class ActionSequenceProcessor:

    def __init__(self, max_len=37):
        self.max_len = max_len
        self.PAD_IDX = 0
        self.action2idx = None
        self.idx2action = None

    # =========================
    # 1. Fit: tạo vocabulary
    # =========================
    def fit(self, X, id_column="id"):

        X_only = X.drop(columns=[id_column]) if id_column in X.columns else X

        all_actions = set()
        for col in X_only.columns:
            values = X_only[col].dropna().astype(int).unique()
            all_actions.update(values)

        all_actions = sorted(list(all_actions))

        self.action2idx = {
            action: idx + 1
            for idx, action in enumerate(all_actions)
        }

        self.idx2action = {
            idx: action
            for action, idx in self.action2idx.items()
        }

        print("Vocabulary size (including PAD):", len(self.action2idx) + 1)
        return self

    # =========================
    # 2. Encode 1 row
    # =========================
    def _encode_row(self, row):
        encoded = []
        for val in row:
            if not np.isnan(val):
                encoded.append(self.action2idx[int(val)])
        return encoded

    # =========================
    # 3. Pad / Truncate
    # =========================
    def _pad_truncate(self, seq):
        if len(seq) > self.max_len:
            return seq[-self.max_len:]
        else:
            return seq + [self.PAD_IDX] * (self.max_len - len(seq))

    # =========================
    # 4. Transform
    # =========================
    def transform(self, X, id_column="id"):

        if self.action2idx is None:
            raise ValueError("Bạn phải fit() trước khi transform()")

        X_only = X.drop(columns=[id_column]) if id_column in X.columns else X

        # Encode
        X_encoded = X_only.apply(
            lambda row: self._encode_row(row),
            axis=1
        )

        # Pad
        X_padded = X_encoded.apply(self._pad_truncate)
        X_padded = np.array(X_padded.tolist())

        # Attention mask
        attention_mask = (X_padded != self.PAD_IDX).astype(int)

        # =============================
        # Statistical features
        # =============================

        seq_length = X_encoded.apply(len)

        unique_ratio = X_encoded.apply(
            lambda seq: len(set(seq)) / len(seq) if len(seq) > 0 else 0
        )

        repetition_ratio = 1 - unique_ratio

        first_action = X_encoded.apply(
            lambda seq: seq[0] if len(seq) > 0 else 0
        )

        last_action = X_encoded.apply(
            lambda seq: seq[-1] if len(seq) > 0 else 0
        )

        X_stat_features = pd.DataFrame({
            "seq_length": seq_length,
            "unique_ratio": unique_ratio,
            "repetition_ratio": repetition_ratio,
            "first_action": first_action,
            "last_action": last_action
        })

        return X_padded, attention_mask, X_stat_features

    # =========================
    # 5. Fit + Transform
    # =========================
    def fit_transform(self, X, id_column="id"):
        self.fit(X, id_column)
        return self.transform(X, id_column)

In [27]:
processor = ActionSequenceProcessor(max_len=37)

# xỬ LÝ DỮ LIỆU TRÊN TẬP X_full_train làm đầu vào cho model để huấn luyện
X_seq_full_train, X_mask_full_train, X_stat_full_train = processor.fit_transform(X_full_train)


Vocabulary size (including PAD): 255


In [28]:
X_seq_full_train.shape, X_mask_full_train.shape, X_stat_full_train.shape

((51000, 37), (51000, 37), (51000, 5))

# =========================================
# BUILD MODEL - SỬ DỤNG BiLSTM + GCN KẾT HỢP STATISTICAL FEATURES
# =========================================
1. BiLSTM: Học tuần tự hành động
2. GCN: Học quan hệ giữa các hành động
3. Statistical Features: Các đặc trưng thống kê từ sequence (đây là các cái nhóm mk bảo thêm ở docs ạ)

In [29]:
# =========================================
#1️⃣ CLASS GCN
# ==================================
import torch
import torch.nn as nn
import torch.nn.functional as F

class ImprovedGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.1):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)
        self.norm = nn.LayerNorm(out_dim)
        self.dropout = nn.Dropout(dropout)
        self.use_residual = (in_dim == out_dim)

    def forward(self, x, adj):
        h = torch.bmm(adj, x)
        h = self.linear(h)

        if self.use_residual:
            h = h + x  # residual đơn giản

        h = self.norm(h)
        h = F.relu(h)
        h = self.dropout(h)
        return h

In [30]:
# ============================================
# 2️⃣ Hybrid BiLSTM + GCN Model
# ============================================
import torch
import torch.nn as nn
import torch.nn.functional as F


class ImprovedBiLSTM_GCN_Model(nn.Module):
    def __init__(
        self,
        num_actions,
        num_classes,
        embed_dim=256,        # Từ 128 → 256
        lstm_hidden=256,       # Từ 128 → 256
        gcn_hidden=256,
        stat_dim=5,            #
        dropout=0.15,
        use_attention=True
    ):
        super().__init__()

        if num_classes is None:
            raise ValueError("num_classes must be provided")

        # =====================
        # 1️⃣ Embedding
        # =====================
        self.embedding = nn.Embedding(
            num_embeddings=num_actions,
            embedding_dim=embed_dim,
            padding_idx=0
        )
        self.embed_dropout = nn.Dropout(dropout)

        # =====================
        # 2️⃣ BiLSTM với dropout
        # =====================
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=lstm_hidden,
            num_layers=5,  # Tăng depth
            batch_first=True,
            bidirectional=True,
            dropout=dropout  # Dropout giữa các layers
        )
        self.lstm_dropout = nn.Dropout(dropout)

        # =====================
        # 3️⃣ Attention (tùy chọn)
        # =====================
        self.use_attention = use_attention
        if use_attention:
            self.attention = nn.MultiheadAttention(
                embed_dim=lstm_hidden * 2,
                num_heads=4,
                dropout=dropout,
                batch_first=True
            )
            self.attn_norm = nn.LayerNorm(lstm_hidden * 2)

        # =====================
        # 4️⃣ GCN Layers (dùng version cải tiến)
        # =====================
        self.gcn1 = ImprovedGCNLayer(embed_dim, gcn_hidden, dropout=dropout)
        self.gcn2 = ImprovedGCNLayer(gcn_hidden, gcn_hidden, dropout=dropout)
        self.gcn3 = ImprovedGCNLayer(gcn_hidden, gcn_hidden, dropout=dropout)
        self.gcn_dropout = nn.Dropout(dropout)

        # =====================
        # 5️⃣ Fusion
        # =====================
        # Tính fusion_dim: lstm_hidden*2 + gcn_hidden + stat_dim
        # 256*2 + 256 + 5 = 512 + 256 + 5 = 773
        fusion_dim = lstm_hidden * 2 + gcn_hidden + stat_dim

        self.fc_shared = nn.Sequential(
            nn.Linear(fusion_dim, 512),  # 773 → 512
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),         # 512 → 256
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # =====================
        # 6️⃣ Multi-head output
        # =====================
        self.heads = nn.ModuleList([
            nn.Linear(256, c) for c in num_classes  # 256 khớp với output fc_shared
        ])

    def build_adj_matrix(self, mask):
        batch_size, seq_len = mask.size()
        device = mask.device

        adj = torch.zeros(batch_size, seq_len, seq_len, device=device)

        # Nối các bước liên tiếp
        for i in range(seq_len - 1):
            adj[:, i, i+1] = 1
            adj[:, i+1, i] = 1

        # Loại bỏ padding
        adj = adj * mask.unsqueeze(1) * mask.unsqueeze(2)

        # Normalize adjacency (quan trọng cho GCN)
        row_sum = adj.sum(dim=-1, keepdim=True) + 1e-8
        adj = adj / row_sum

        return adj

    def forward(self, x_seq, mask, stat_feat):
        # =====================
        # Embedding
        # =====================
        emb = self.embedding(x_seq)  # (B, L, D)
        emb = self.embed_dropout(emb)

        # =====================
        # BiLSTM branch
        # =====================
        lstm_out, _ = self.lstm(emb)  # (B, L, 2H)
        lstm_out = lstm_out * mask.unsqueeze(-1)

        # Attention (nếu có)
        if self.use_attention:
            attn_out, _ = self.attention(lstm_out, lstm_out, lstm_out, key_padding_mask=(1-mask).bool())
            lstm_out = self.attn_norm(lstm_out + attn_out)  # Residual connection

        lstm_out = self.lstm_dropout(lstm_out)

        # Mean pooling
        lstm_feat = torch.sum(lstm_out, dim=1) / (mask.sum(dim=1, keepdim=True) + 1e-8)

        # =====================
        # GCN branch
        # =====================
        adj = self.build_adj_matrix(mask)

        gcn_out = self.gcn1(emb, adj)
        gcn_out = self.gcn2(gcn_out, adj)
        gcn_out = self.gcn3(gcn_out, adj)  # Thêm GCN layer thứ 3
        gcn_out = gcn_out * mask.unsqueeze(-1)
        gcn_out = self.gcn_dropout(gcn_out)

        gcn_feat = torch.sum(gcn_out, dim=1) / (mask.sum(dim=1, keepdim=True) + 1e-8)

        # =====================
        # Fusion
        # =====================
        fused = torch.cat([lstm_feat, gcn_feat, stat_feat], dim=1)  # (B, 773)
        shared = self.fc_shared(fused)  # (B, 256)

        # =====================
        # Multi-output heads
        # =====================
        outputs = [head(shared) for head in self.heads]  # Mỗi head cho 1 attr

        return outputs

In [31]:
num_classes = [
    len(encoders[col].classes_)
    for col in Y_full_train.columns
]

In [32]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

def focal_loss(logits, targets, gamma=2.0):
    """
    Focal Loss đơn giản (không class weights)
    - gamma=2.0: tập trung vào mẫu khó
    """
    ce_loss = F.cross_entropy(logits, targets, reduction='none')
    pt = torch.exp(-ce_loss)
    focal_loss = ((1 - pt) ** gamma) * ce_loss
    return focal_loss.mean()


def consistency_loss(pred_1, pred_4, valid_combinations):
    """
    Phạt khi model dự đoán cặp (attr_1, attr_4) không hợp lệ
    """
    pred_1 = torch.argmax(pred_1, dim=1)
    pred_4 = torch.argmax(pred_4, dim=1)

    valid_set = torch.tensor(valid_combinations, device=pred_1.device)
    batch_size = pred_1.size(0)
    invalid_count = 0

    for i in range(batch_size):
        pair = torch.tensor([pred_1[i], pred_4[i]], device=pred_1.device)
        is_valid = ((valid_set[:, 0] == pair[0]) & (valid_set[:, 1] == pair[1])).any()
        if not is_valid:
            invalid_count += 1

    return 0.1 * (invalid_count / batch_size)


def multi_output_loss_with_focal(
    outputs,
    targets,
    valid_combinations=None
):
    """
    LOSS FUNCTION: Focal cho attr_1,4 - CrossEntropy cho attr còn lại
    """
    total_loss = 0
    loss_details = {}

    for i in range(6):
        if i in [0, 3]:  # ATTR_1 và ATTR_4 - DÙNG FOCAL LOSS
            loss = focal_loss(outputs[i], targets[:, i], gamma=1.5)
            loss = loss * 1.5  # Ưu tiên nhẹ
        else:  # Các attr còn lại - CrossEntropy thường
            loss = F.cross_entropy(outputs[i], targets[:, i])

        total_loss += loss
        loss_details[f'attr_{i+1}'] = loss.item()

    # Consistency loss cho cặp (attr_1, attr_4)
    if valid_combinations is not None and len(valid_combinations) > 0:
        cons_loss = consistency_loss(outputs[0], outputs[3], valid_combinations)
        total_loss += cons_loss
        loss_details['consistency'] = cons_loss

    return total_loss, loss_details

In [33]:
# 📈 Exact Match Accuracy
def exact_match_accuracy(outputs, targets):

    preds = [torch.argmax(out, dim=1) for out in outputs]
    preds = torch.stack(preds, dim=1)

    correct = (preds == targets).all(dim=1).float()
    return correct.mean().item()

In [34]:
# 1️⃣ Tạo Dataset
from torch.utils.data import Dataset

class BehaviorDataset(Dataset):
    def __init__(self, X_seq, X_mask, X_stat, Y):
        self.X_seq = torch.tensor(X_seq, dtype=torch.long)
        self.X_mask = torch.tensor(X_mask, dtype=torch.float32)
        self.X_stat = torch.tensor(X_stat.values, dtype=torch.float32)
        self.Y = torch.tensor(Y.values, dtype=torch.long)

    def __len__(self):
        return len(self.Y)

    def __getitem__(self, idx):
        return (
            self.X_seq[idx],
            self.X_mask[idx],
            self.X_stat[idx],
            self.Y[idx]
        )

In [35]:
# ====================
# Đây là dữ liệu dùng để train
# ===================

from torch.utils.data import DataLoader

train_dataset = BehaviorDataset(
    X_seq_full_train,
    X_mask_full_train,
    X_stat_full_train,
    Y_train_encoded
)

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True
)

In [36]:
# import torch

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# print("Using device:", device)
device = torch.device("cpu")

In [37]:
print("X_seq:", len(X_seq_full_train))
print("Mask:", len(X_mask_full_train))
print("Stat:", len(X_stat_full_train))
print("Y:", len(Y_train_encoded))

X_seq: 51000
Mask: 51000
Stat: 51000
Y: 51000


In [38]:
# =====================
# TRANSFORM X_VAL : Đoạn này chuẩn bị dữ liệu validation để tính độ chính xác (Val Acc) sau mỗi epoch ạ
# =====================

# Dùng processor đã fit từ train để transform X_val
X_seq_val, X_mask_val, X_stat_val = processor.transform(X_val)

print("X_seq_val shape:", X_seq_val.shape)
print("X_mask_val shape:", X_mask_val.shape)
print("X_stat_val shape:", X_stat_val.shape)

X_seq_val shape: (7200, 37)
X_mask_val shape: (7200, 37)
X_stat_val shape: (7200, 5)


In [39]:
# =====================
# TẠO VALIDATION DATASET:
# =====================


# 1. Tách riêng ID và features
id_val = Y_val['id'] if 'id' in Y_val.columns else None

# 2. Chỉ lấy 6 cột attr để encode
feature_cols = [col for col in Y_val.columns if col.startswith('attr_')]
Y_val_features = Y_val[feature_cols]

# 3. Encode validation (dùng encoder đã fit từ train)
Y_val_encoded = Y_val_features.copy()
for col in feature_cols:
    Y_val_encoded[col] = encoders[col].transform(Y_val_features[col])

print("Y_val_encoded shape:", Y_val_encoded.shape)  # (samples, 6)
print("Columns:", Y_val_encoded.columns.tolist())

# 4. Tạo validation dataset
val_dataset = BehaviorDataset(
    X_seq_val,
    X_mask_val,
    X_stat_val,
    Y_val_encoded
)

val_loader = DataLoader(
    val_dataset,
    batch_size=256,
    shuffle=False
)

Y_val_encoded shape: (7200, 6)
Columns: ['attr_1', 'attr_2', 'attr_3', 'attr_4', 'attr_5', 'attr_6']


In [ ]:
# =====================
# Training
# =====================

import pickle
drive_path = '/content/drive/MyDrive/ModelBiLSTM/'

# TÍNH VALID COMBINATIONS CHO (attr_1, attr_4)
a1_vals = Y_full_train['attr_1'].values
a4_vals = Y_full_train['attr_4'].values
valid_pairs = set(zip(a1_vals, a4_vals))
valid_combinations = [[x-1, y-1] for x, y in valid_pairs]
print(f"✅ Tìm thấy {len(valid_combinations)} cặp (attr_1, attr_4) hợp lệ")
print(f"   Ví dụ 5 cặp đầu: {valid_combinations[:5]}")

# Xác định device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device}")

# Tạo model
model = ImprovedBiLSTM_GCN_Model(
    num_actions=len(processor.action2idx) + 1,
    num_classes=num_classes,
    dropout=0.15,
    use_attention=False
).to(device)

print(f"✅ Đã tạo model với {sum(p.numel() for p in model.parameters()):,} parameters")

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

best_val_acc = 0
patience_counter = 0
patience = 5

for epoch in range(50):
    # ===== TRAIN =====
    model.train()
    train_loss = 0
    train_acc = 0

    for x_seq, mask, stat_feat, targets in train_loader:
        x_seq = x_seq.to(device)
        mask = mask.to(device)
        stat_feat = stat_feat.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        outputs = model(x_seq, mask, stat_feat)

        loss, loss_details = multi_output_loss_with_focal(
            outputs,
            targets,
            valid_combinations=valid_combinations
        )
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item()
        train_acc += exact_match_accuracy(outputs, targets)

    # ===== VALIDATION =====
    model.eval()
    val_loss = 0
    val_acc = 0

    with torch.no_grad():
        for x_seq, mask, stat_feat, targets in val_loader:
            x_seq = x_seq.to(device)
            mask = mask.to(device)
            stat_feat = stat_feat.to(device)
            targets = targets.to(device)

            outputs = model(x_seq, mask, stat_feat)
            loss, _ = multi_output_loss_with_focal(
                outputs,
                targets,
                valid_combinations=valid_combinations
            )
            acc = exact_match_accuracy(outputs, targets)

            val_loss += loss.item()
            val_acc += acc

    # ===== LOGGING =====
    train_loss /= len(train_loader)
    train_acc /= len(train_loader)
    val_loss /= len(val_loader)
    val_acc /= len(val_loader)

    # In kết quả epoch
    print(f"\nEpoch {epoch+1}:")
    print(f"   Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"   Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    print(f"   LR: {optimizer.param_groups[0]['lr']:.2e}")

    if epoch == 0:
        print(f"   Loss details: {loss_details}")

    # ===== SCHEDULER =====
    scheduler.step(val_loss)

    # ===== EARLY STOPPING & SAVE =====
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0

        # Lưu model
        model_path = drive_path + "best_model.pth"
        torch.save(model.state_dict(), model_path)

        # Lưu encoders
        with open(drive_path + "label_encoders.pkl", "wb") as f:
            pickle.dump(encoders, f)

        # Lưu processor
        with open(drive_path + "processor.pkl", "wb") as f:
            pickle.dump(processor, f)

        print(f"   ✅ ĐÃ LƯU MODEL TỐT NHẤT!")
        print(f"      - Val Acc: {val_acc:.4f} (tốt hơn {best_val_acc:.4f})")
        print(f"      - Đường dẫn: {model_path}")

        # Lưu thêm checkpoint để phòng restart
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_acc': best_val_acc,
            'patience_counter': patience_counter
        }
        torch.save(checkpoint, drive_path + "checkpoint_last.pth")
    else:
        patience_counter += 1
        print(f"   Patience: {patience_counter}/{patience}")

        if patience_counter >= patience:
            print(f"\n🎉 Early stopping tại epoch {epoch+1}")
            print(f"   Best Val Acc đạt được: {best_val_acc:.4f}")
            break

# Kết thúc training
print("\n" + "="*50)
print("🎯 KẾT THÚC TRAINING")
print("="*50)
print(f"🏆 Best Validation Accuracy: {best_val_acc:.4f}")
print(f"💾 Model đã lưu tại: {drive_path}best_model.pth")
print(f"💾 Encoders đã lưu tại: {drive_path}label_encoders.pkl")
print(f"💾 Processor đã lưu tại: {drive_path}processor.pkl")

# Load best model for testing
model.load_state_dict(torch.load(drive_path + "best_model.pth"))
print(f"✅ Đã load best model (Val Acc: {best_val_acc:.4f})")

✅ Tìm thấy 74 cặp (attr_1, attr_4) hợp lệ
   Ví dụ 5 cặp đầu: [[np.int64(11), np.int64(0)], [np.int64(2), np.int64(0)], [np.int64(3), np.int64(5)], [np.int64(4), np.int64(0)], [np.int64(8), np.int64(4)]]
✅ Using device: cpu
✅ Đã tạo model với 8,225,308 parameters


#### Dự đoán